In [5]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

TMDB_API_KEY = os.getenv("TMDB_API_KEY")
TMDB_ACCESS_TOKEN = os.getenv("TMDB_ACCESS_TOKEN")

print("API Key loaded:", bool(TMDB_API_KEY))
print("Access Token loaded:", bool(TMDB_ACCESS_TOKEN))

API Key loaded: True
Access Token loaded: True


In [6]:
import requests

url = "https://api.themoviedb.org/3/movie/popular"

headers = {
    "Authorization": f"Bearer {TMDB_ACCESS_TOKEN}",
    "accept": "application/json"
}

params = {
    "language": "en-US",
    "page": 1
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status Code:", response.status_code)

Status Code: 200


In [12]:
import time
import requests

def tmdb_request(endpoint, params=None, retries=3):
    
    base_url = "https://api.themoviedb.org/3"
    
    for attempt in range(1, retries + 1):
        
        try:
            response = requests.get(
                base_url + endpoint,
                headers=headers,
                params=params,
                timeout=20
            )
            
            if response.status_code == 200:
                return response.json()
            
            print(f"HTTP Error {response.status_code}")
            
            if response.status_code == 429:
                print("Rate limit reached. Waiting...")
                time.sleep(5)
            else:
                return None
        
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt}/{retries} failed: {type(e).__name__}")
            
            if attempt < retries:
                time.sleep(3)
            else:
                print("Request failed after all retries.")
    
    return None

In [13]:
def fetch_movies(num_pages=10):
    
    movies = []
    
    for page in range(1, num_pages + 1):
        
        data = tmdb_request(
            "/discover/movie",
            {
                "language": "en-US",
                "sort_by": "popularity.desc",
                "page": page
            }
        )
        
        if data is None:
            print(f"Skipping page {page}")
            continue
        
        movies.extend(data["results"])
        
        print(f"Page {page}/{num_pages} collected | Total: {len(movies)}")
        
        # Small delay between requests
        time.sleep(1)
    
    return movies

In [14]:
movies = fetch_movies(3)

print("Total movies collected:", len(movies))

Page 1/3 collected | Total: 20
Page 2/3 collected | Total: 40
Page 3/3 collected | Total: 60
Total movies collected: 60


In [20]:
import pandas as pd
movies_df = pd.DataFrame(movies)

print("Shape:", movies_df.shape)

Shape: (60, 15)


In [21]:
movies_df.head()

,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/qeQJx07rK2xm8SD2sJxFKhE7gs0.jpg,"[878, 28, 12]",969681,Spider-Man: Brand New Day,en,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,1065.0058,/iPOn6DinuVyLY17YM9mKuPofV08.jpg,2026-07-29,False,False,7.863,1620
1,False,/RMXG8myu1aGlNUsRjtxzmpdMK0.jpg,"[12, 28, 14]",1368337,The Odyssey,en,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",862.2605,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,2026-07-15,False,False,7.995,2682
2,False,/14QbnygCuTO0vl7CAFmPf1fgZfV.jpg,"[28, 12, 878]",634649,Spider-Man: No Way Home,en,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,634.4715,/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg,2021-12-15,False,False,7.938,22629
3,False,/2mPmccLg8QCD4ZF6v8kSsUijPPW.jpg,"[27, 878, 53]",1284041,The Last House,en,The Last House,A family suddenly sealed inside their home mus...,388.6318,/6JU7E8Vv2M11egkctWVOScxWR75.jpg,2026-08-06,False,False,6.865,519
4,False,/kkcwhgSFd81QDlXo8ytrpHPQjhy.jpg,"[12, 16, 35, 10751, 14]",1315772,Minions & Monsters,en,Minions & Monsters,"This is the rambunctious, ridiculous and total...",344.0984,/nz7i42yhLIJ4ve9JKgM6NthoLHO.jpg,2026-06-24,False,False,7.226,488


In [22]:
movies_df.columns.tolist()

['adult',
 'backdrop_path',
 'genre_ids',
 'id',
 'title',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'poster_path',
 'release_date',
 'softcore',
 'video',
 'vote_average',
 'vote_count']

In [23]:
movies_df.isnull().sum()

adult                0
backdrop_path        1
genre_ids            0
id                   0
title                0
original_language    0
original_title       0
overview             0
popularity           0
poster_path          0
release_date         0
softcore             0
video                0
vote_average         0
vote_count           0
dtype: int64

In [24]:
genre_data = tmdb_request(
    "/genre/movie/list",
    {
        "language": "en-US"
    }
)

genre_data.keys()

dict_keys(['genres'])

In [25]:
genre_data["genres"][:5]

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 16, 'name': 'Animation'},
 {'id': 35, 'name': 'Comedy'},
 {'id': 80, 'name': 'Crime'}]

In [26]:
genre_map = {
    genre["id"]: genre["name"]
    for genre in genre_data["genres"]
}

genre_map

{28: 'Action',
 12: 'Adventure',
 16: 'Animation',
 35: 'Comedy',
 80: 'Crime',
 99: 'Documentary',
 18: 'Drama',
 10751: 'Family',
 14: 'Fantasy',
 36: 'History',
 27: 'Horror',
 10402: 'Music',
 9648: 'Mystery',
 10749: 'Romance',
 878: 'Science Fiction',
 10770: 'TV Movie',
 53: 'Thriller',
 10752: 'War',
 37: 'Western'}

In [28]:
movies_df["genres"] = movies_df["genre_ids"].apply(
    lambda ids: [genre_map.get(genre_id) for genre_id in ids]
)
movies_df[["title", "genre_ids", "genres"]].head()

,title,genre_ids,genres
0,Spider-Man: Brand New Day,"[878, 28, 12]","[Science Fiction, Action, Adventure]"
1,The Odyssey,"[12, 28, 14]","[Adventure, Action, Fantasy]"
2,Spider-Man: No Way Home,"[28, 12, 878]","[Action, Adventure, Science Fiction]"
3,The Last House,"[27, 878, 53]","[Horror, Science Fiction, Thriller]"
4,Minions & Monsters,"[12, 16, 35, 10751, 14]","[Adventure, Animation, Comedy, Family, Fantasy]"


In [29]:
movie_id = int(movies_df.iloc[0]["id"])

print("Movie:", movies_df.iloc[0]["title"])
print("ID:", movie_id)

Movie: Spider-Man: Brand New Day
ID: 969681


In [30]:
details = tmdb_request(
    f"/movie/{movie_id}",
    {
        "language": "en-US"
    }
)

print("Status:", "Success" if details else "Failed")

Attempt 1/3 failed: ConnectionError
Status: Success


In [31]:
details.keys()

dict_keys(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count'])

In [32]:
credits = tmdb_request(
    f"/movie/{movie_id}/credits",
    {
        "language": "en-US"
    }
)

print("Status:", "Success" if credits else "Failed")

Status: Success


In [33]:
credits.keys()

dict_keys(['id', 'cast', 'crew'])

In [34]:
credits["cast"][:5]

[{'adult': False,
  'gender': 2,
  'id': 1136406,
  'known_for_department': 'Acting',
  'name': 'Tom Holland',
  'original_name': 'Tom Holland',
  'popularity': 32.4982,
  'profile_path': '/xKBAaPIa1c7tzZD3Y0MhBLv4hPE.jpg',
  'cast_id': 4,
  'character': 'Peter Parker / Spider-Man',
  'credit_id': '630cbd43ede1b00083c3badf',
  'order': 0},
 {'adult': False,
  'gender': 1,
  'id': 505710,
  'known_for_department': 'Acting',
  'name': 'Zendaya',
  'original_name': 'Zendaya',
  'popularity': 19.3523,
  'profile_path': '/3WdOloHpjtjL96uVOhFRRCcYSwq.jpg',
  'cast_id': 49,
  'character': 'MJ',
  'credit_id': '6855e165f57c8fc6a7ae16ed',
  'order': 1},
 {'adult': False,
  'gender': 2,
  'id': 103,
  'known_for_department': 'Acting',
  'name': 'Mark Ruffalo',
  'original_name': 'Mark Ruffalo',
  'popularity': 7.8723,
  'profile_path': '/5GilHMOt5PAQh6rlUKZzGmaKEI7.jpg',
  'cast_id': 55,
  'character': 'Bruce Banner / Hulk',
  'credit_id': '688d2b3704c5bd1f52de627a',
  'order': 2},
 {'adult': Fa

In [35]:
credits["crew"][:5]

[{'adult': False,
  'gender': 2,
  'id': 10850,
  'known_for_department': 'Production',
  'name': 'Kevin Feige',
  'original_name': 'Kevin Feige',
  'popularity': 3.706,
  'profile_path': '/kCBqXZ5PT5udYGEj2wfTSFbLMvT.jpg',
  'credit_id': '628d559ed48cee2cbfca858a',
  'department': 'Production',
  'job': 'Producer'},
 {'adult': False,
  'gender': 1,
  'id': 1425616,
  'known_for_department': 'Production',
  'name': 'Amy Pascal',
  'original_name': 'Amy Pascal',
  'popularity': 2.1615,
  'profile_path': '/cZ526e4oPN1i44Q0BxYW8nOtrzR.jpg',
  'credit_id': '65e51be61ad93b01860898a0',
  'department': 'Production',
  'job': 'Producer'},
 {'adult': False,
  'gender': 2,
  'id': 57027,
  'known_for_department': 'Production',
  'name': "Louis D'Esposito",
  'original_name': "Louis D'Esposito",
  'popularity': 2.838,
  'profile_path': '/ewtzT2ZMIX2DNyGUUkHLn36fjgF.jpg',
  'credit_id': '641a69d3688cd0010c5f38d8',
  'department': 'Production',
  'job': 'Executive Producer'},
 {'adult': False,
  'g

In [37]:
keyword_data = tmdb_request(
    f"/movie/{movie_id}/keywords"
)

print("Status:", "Success" if keyword_data else "Failed")

Status: Success


In [38]:
keyword_data["keywords"]

[{'id': 9678, 'name': 'mind control'},
 {'id': 242, 'name': 'new york city'},
 {'id': 1701, 'name': 'hero'},
 {'id': 2766, 'name': 'mutation'},
 {'id': 1308, 'name': 'secret identity'},
 {'id': 9715, 'name': 'superhero'},
 {'id': 3986, 'name': 'spider'},
 {'id': 3289, 'name': 'villain'},
 {'id': 9717, 'name': 'based on comic'},
 {'id': 9663, 'name': 'sequel'},
 {'id': 12375, 'name': 'transhumanism'},
 {'id': 33637, 'name': 'super power'},
 {'id': 158456, 'name': 'masked vigilante'},
 {'id': 169909, 'name': 'spider web'},
 {'id': 179430, 'name': 'aftercreditsstinger'},
 {'id': 180547, 'name': 'marvel cinematic universe (mcu)'},
 {'id': 180734, 'name': 'masked superhero'},
 {'id': 193946, 'name': 'fight for justice'},
 {'id': 233300, 'name': 'genetic mutation'}]

In [39]:
cast_names = [
    person["name"]
    for person in credits["cast"][:10]
]

print(cast_names)

['Tom Holland', 'Zendaya', 'Mark Ruffalo', 'Jon Bernthal', 'Jacob Batalon', 'Sadie Sink', 'Florence Pugh', 'Liza Colón-Zayas', 'Tramell Tillman', 'Marisa Tomei']


In [40]:
directors = [
    person["name"]
    for person in credits["crew"]
    if person["job"] == "Director"
]

print(directors)

['Destin Daniel Cretton']


In [41]:
keyword_names = [
    keyword["name"]
    for keyword in keyword_data["keywords"]
]

print(keyword_names)

['mind control', 'new york city', 'hero', 'mutation', 'secret identity', 'superhero', 'spider', 'villain', 'based on comic', 'sequel', 'transhumanism', 'super power', 'masked vigilante', 'spider web', 'aftercreditsstinger', 'marvel cinematic universe (mcu)', 'masked superhero', 'fight for justice', 'genetic mutation']


In [42]:
def enrich_movie(movie_id):
    
    # 1. Get movie details
    details = tmdb_request(
        f"/movie/{movie_id}",
        {
            "language": "en-US"
        }
    )
    
    if details is None:
        return None
    
    
    # 2. Get credits
    credits = tmdb_request(
        f"/movie/{movie_id}/credits",
        {
            "language": "en-US"
        }
    )
    
    
    # 3. Get keywords
    keyword_data = tmdb_request(
        f"/movie/{movie_id}/keywords"
    )
    
    
    # 4. Extract genres
    genres = [
        genre["name"]
        for genre in details.get("genres", [])
    ]
    
    
    # 5. Extract top 10 cast members
    cast = [
        person["name"]
        for person in credits.get("cast", [])[:10]
    ] if credits else []
    
    
    # 6. Extract director
    directors = [
        person["name"]
        for person in credits.get("crew", [])
        if person.get("job") == "Director"
    ] if credits else []
    
    
    # 7. Extract keywords
    keywords = [
        keyword["name"]
        for keyword in keyword_data.get("keywords", [])
    ] if keyword_data else []
    
    
    # 8. Create final movie dictionary
    movie = {
        "id": details.get("id"),
        "title": details.get("title"),
        "overview": details.get("overview"),
        "genres": genres,
        "keywords": keywords,
        "cast": cast,
        "director": directors,
        "runtime": details.get("runtime"),
        "release_date": details.get("release_date"),
        "vote_average": details.get("vote_average"),
        "vote_count": details.get("vote_count"),
        "popularity": details.get("popularity"),
        "poster_path": details.get("poster_path"),
        "backdrop_path": details.get("backdrop_path"),
        "original_language": details.get("original_language")
    }
    
    return movie

In [46]:
movie_data = enrich_movie(movie_id)
print("Title:", movie_data["title"])
print("Genres:", movie_data["genres"])
print("Keywords:", movie_data["keywords"])
print("Cast:", movie_data["cast"])
print("Director:", movie_data["director"])
print("Runtime:", movie_data["runtime"])
print("Rating:", movie_data["vote_average"])

Title: Spider-Man: Brand New Day
Genres: ['Science Fiction', 'Action', 'Adventure']
Keywords: ['mind control', 'new york city', 'hero', 'mutation', 'secret identity', 'superhero', 'spider', 'villain', 'based on comic', 'sequel', 'transhumanism', 'super power', 'masked vigilante', 'spider web', 'aftercreditsstinger', 'marvel cinematic universe (mcu)', 'masked superhero', 'fight for justice', 'genetic mutation']
Cast: ['Tom Holland', 'Zendaya', 'Mark Ruffalo', 'Jon Bernthal', 'Jacob Batalon', 'Sadie Sink', 'Florence Pugh', 'Liza Colón-Zayas', 'Tramell Tillman', 'Marisa Tomei']
Director: ['Destin Daniel Cretton']
Runtime: 145
Rating: 7.859


In [47]:
import os
import json
import time

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

print("Data folders ready.")

Data folders ready.


In [48]:
enriched_movies = []

for i, movie in enumerate(movies, start=1):
    
    movie_id = movie["id"]
    title = movie["title"]
    
    print(f"[{i}/{len(movies)}] Processing: {title}")
    
    try:
        enriched = enrich_movie(movie_id)
        
        if enriched is not None:
            enriched_movies.append(enriched)
        
        # Small delay to avoid hitting the API too aggressively
        time.sleep(1)
        
    except Exception as e:
        print(f"❌ Failed: {title} | {e}")
    
    # Save checkpoint every 10 movies
    if i % 10 == 0:
        
        with open(
            "../data/raw/enriched_movies_checkpoint.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(enriched_movies, f, ensure_ascii=False, indent=2)
        
        print(f"💾 Checkpoint saved: {len(enriched_movies)} movies")

[1/60] Processing: Spider-Man: Brand New Day
[2/60] Processing: The Odyssey
[3/60] Processing: Spider-Man: No Way Home
[4/60] Processing: The Last House
[5/60] Processing: Minions & Monsters
[6/60] Processing: Colony
[7/60] Processing: Toy Story 5
[8/60] Processing: Obsession
[9/60] Processing: Supergirl
[10/60] Processing: Evil Dead Burn
💾 Checkpoint saved: 10 movies
[11/60] Processing: The Death of Robin Hood
[12/60] Processing: Moana
[13/60] Processing: Disclosure Day
[14/60] Processing: Spider-Man: Homecoming
[15/60] Processing: 仙逆剧场版 弑仙之战
[16/60] Processing: The Devil's Mouth
[17/60] Processing: Soulm8te
[18/60] Processing: Backrooms
[19/60] Processing: Spider-Man
[20/60] Processing: Scary Movie
💾 Checkpoint saved: 20 movies
[21/60] Processing: The Odyssey
[22/60] Processing: Masters of the Universe
[23/60] Processing: Avatar Aang: The Last Airbender
[24/60] Processing: Project Hail Mary
[25/60] Processing: Leviticus
[26/60] Processing: Demon Slayer: Kimetsu no Yaiba Infinity Cast

In [50]:
print("Movies successfully enriched:", len(enriched_movies))

Movies successfully enriched: 60


In [51]:
enriched_df = pd.DataFrame(enriched_movies)

print("Shape:", enriched_df.shape)

Shape: (60, 15)


In [52]:
enriched_df.head()

,id,title,overview,genres,keywords,cast,director,runtime,release_date,vote_average,vote_count,popularity,poster_path,backdrop_path,original_language
0,969681,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,"[Science Fiction, Action, Adventure]","[mind control, new york city, hero, mutation, ...","[Tom Holland, Zendaya, Mark Ruffalo, Jon Bernt...",[Destin Daniel Cretton],145,2026-07-29,7.859,1618,1065.0058,/iPOn6DinuVyLY17YM9mKuPofV08.jpg,/qeQJx07rK2xm8SD2sJxFKhE7gs0.jpg,en
1,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...","[Adventure, Action, Fantasy]","[ship, trojan war, greek mythology, historical...","[Matt Damon, Tom Holland, Anne Hathaway, Rober...",[Christopher Nolan],173,2026-07-15,8.000,2681,862.2605,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,/RMXG8myu1aGlNUsRjtxzmpdMK0.jpg,en
2,634649,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,"[Action, Adventure, Science Fiction]","[new york city, hero, showdown, magic, loss of...","[Tom Holland, Zendaya, Benedict Cumberbatch, J...",[Jon Watts],148,2021-12-15,7.938,22633,634.4715,/1g0dhYtq4irTY1GPXvft6k4YLjm.jpg,/14QbnygCuTO0vl7CAFmPf1fgZfV.jpg,en
3,1284041,The Last House,A family suddenly sealed inside their home mus...,"[Horror, Science Fiction, Thriller]","[monster, water monster, sea creature, caution...","[Greta Lee, Wagner Moura, Riley Chung, Noah Al...",[Louis Leterrier],110,2026-08-06,6.871,520,388.6318,/6JU7E8Vv2M11egkctWVOScxWR75.jpg,/2mPmccLg8QCD4ZF6v8kSsUijPPW.jpg,en
4,1315772,Minions & Monsters,"This is the rambunctious, ridiculous and total...","[Adventure, Animation, Comedy, Family, Fantasy]","[magic, sequel, alien, prequel, hollywood, spi...","[Pierre Coffin, Trey Parker, Christoph Waltz, ...",[Pierre Coffin],90,2026-06-24,7.209,485,344.0984,/nz7i42yhLIJ4ve9JKgM6NthoLHO.jpg,/kkcwhgSFd81QDlXo8ytrpHPQjhy.jpg,en


In [53]:
enriched_df.to_json(
    "../data/processed/movies_enriched.json",
    orient="records",
    indent=2,
    force_ascii=False
)

print("✅ Enriched dataset saved.")

✅ Enriched dataset saved.


In [54]:
enriched_df.to_csv(
    "../data/processed/movies_enriched.csv",
    index=False
)

print("✅ CSV saved.")

✅ CSV saved.


In [55]:
enriched_df[[
    "title",
    "overview",
    "genres",
    "keywords",
    "cast",
    "director"
]].head()

,title,overview,genres,keywords,cast,director
0,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,"[Science Fiction, Action, Adventure]","[mind control, new york city, hero, mutation, ...","[Tom Holland, Zendaya, Mark Ruffalo, Jon Bernt...",[Destin Daniel Cretton]
1,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...","[Adventure, Action, Fantasy]","[ship, trojan war, greek mythology, historical...","[Matt Damon, Tom Holland, Anne Hathaway, Rober...",[Christopher Nolan]
2,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,"[Action, Adventure, Science Fiction]","[new york city, hero, showdown, magic, loss of...","[Tom Holland, Zendaya, Benedict Cumberbatch, J...",[Jon Watts]
3,The Last House,A family suddenly sealed inside their home mus...,"[Horror, Science Fiction, Thriller]","[monster, water monster, sea creature, caution...","[Greta Lee, Wagner Moura, Riley Chung, Noah Al...",[Louis Leterrier]
4,Minions & Monsters,"This is the rambunctious, ridiculous and total...","[Adventure, Animation, Comedy, Family, Fantasy]","[magic, sequel, alien, prequel, hollywood, spi...","[Pierre Coffin, Trey Parker, Christoph Waltz, ...",[Pierre Coffin]


In [56]:
enriched_df[[
    "overview",
    "genres",
    "keywords",
    "cast",
    "director"
]].isna().sum()

overview    0
genres      0
keywords    0
cast        0
director    0
dtype: int64

In [57]:
def convert_to_text(value):
    if isinstance(value, list):
        return " ".join(value)
    return str(value)

In [58]:
text_columns = [
    "overview",
    "genres",
    "keywords",
    "cast",
    "director"
]

for column in text_columns:
    enriched_df[column] = enriched_df[column].apply(convert_to_text)

In [59]:
enriched_df["combined_text"] = (
    enriched_df["overview"] + " " +
    enriched_df["genres"] + " " +
    enriched_df["keywords"] + " " +
    enriched_df["cast"] + " " +
    enriched_df["director"]
)

In [65]:
print(enriched_df[["title", "combined_text"]].iloc[0])

title                                    Spider-Man: Brand New Day
combined_text    Fighting crime full-time as Spider-Man in a wo...
Name: 0, dtype: str


In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("TF-IDF ready!")

TF-IDF ready!


In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(enriched_df["combined_text"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (60, 2403)


In [68]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:", similarity_matrix.shape)

Similarity Matrix Shape: (60, 60)


In [70]:
def recommend_movies(movie_title, top_n=5):
    
    # Find movie index
    movie_indices = enriched_df[
        enriched_df["title"].str.lower() == movie_title.lower()
    ].index
    
    if len(movie_indices) == 0:
        print("Movie not found.")
        return
    
    movie_index = movie_indices[0]
    
    # Get similarity scores
    similarity_scores = list(enumerate(similarity_matrix[movie_index]))
    
    # Sort by similarity score
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    # Remove the movie itself
    similarity_scores = similarity_scores[1:top_n + 1]
    
    # Get recommended movies
    recommendations = []
    
    for index, score in similarity_scores:
        
        recommendations.append({
            "title": enriched_df.iloc[index]["title"],
            "similarity_score": round(score * 100, 2)
        })
    
    return pd.DataFrame(recommendations)

In [71]:
recommend_movies("Spider-Man: Brand New Day")

,title,similarity_score
0,Spider-Man: No Way Home,33.05
1,Spider-Man: Homecoming,32.03
2,The Amazing Spider-Man 2,24.08
3,Spider-Man: Far From Home,23.46
4,Spider-Man,21.69


In [72]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def recommend_for_user(liked_movies, top_n=5):
    
    movie_indices = []
    
    # Find indexes of liked movies
    for movie in liked_movies:
        
        matches = enriched_df[
            enriched_df["title"].str.lower() == movie.lower()
        ].index
        
        if len(matches) > 0:
            movie_indices.append(matches[0])
        else:
            print(f"Movie not found: {movie}")
    
    
    if len(movie_indices) == 0:
        print("No valid movies found.")
        return
    
    
    # Get TF-IDF vectors of liked movies
    liked_vectors = tfidf_matrix[movie_indices]
    
    
    # Create user taste vector
    user_vector = liked_vectors.mean(axis=0)
    
    
    # Compare user taste with every movie
    user_similarity = cosine_similarity(
        user_vector,
        tfidf_matrix
    ).flatten()
    
    
    # Rank movies
    ranked_indices = np.argsort(user_similarity)[::-1]
    
    
    recommendations = []
    
    for index in ranked_indices:
        
        # Don't recommend movies the user already likes
        if index in movie_indices:
            continue
        
        recommendations.append({
            "title": enriched_df.iloc[index]["title"],
            "similarity_score": round(user_similarity[index] * 100, 2)
        })
        
        if len(recommendations) == top_n:
            break
    
    
    return pd.DataFrame(recommendations)

In [73]:
recommend_for_user(
    [
        "Interstellar",
        "Inception",
        "The Martian"
    ],
    top_n=5
)

Movie not found: Interstellar
Movie not found: Inception
Movie not found: The Martian
No valid movies found.


In [74]:
enriched_df["title"].tolist()

['Spider-Man: Brand New Day',
 'The Odyssey',
 'Spider-Man: No Way Home',
 'The Last House',
 'Minions & Monsters',
 'Colony',
 'Toy Story 5',
 'Obsession',
 'Supergirl',
 'Evil Dead Burn',
 'The Death of Robin Hood',
 'Moana',
 'Disclosure Day',
 'Spider-Man: Homecoming',
 '仙逆剧场版 弑仙之战',
 "The Devil's Mouth",
 'Soulm8te',
 'Backrooms',
 'Spider-Man',
 'Scary Movie',
 'The Odyssey',
 'Masters of the Universe',
 'Avatar Aang: The Last Airbender',
 'Project Hail Mary',
 'Leviticus',
 'Demon Slayer: Kimetsu no Yaiba Infinity Castle',
 'Lucky Strike',
 'Spider-Man: Far From Home',
 'The Debt Collector',
 'Borderline',
 'The Novices',
 'The Devil Wears Prada 2',
 'Kraken',
 'The Amazing Spider-Man',
 'Shake, Rattle & Roll: Evil Origins',
 'The Invite',
 'The Isolate Thief',
 'Michael',
 'Master of the Universe',
 'Your Heart Will Be Broken',
 'Avengers: Infinity War',
 'Hotel Desire',
 'Deep Water',
 'The Amazing Spider-Man 2',
 'Mortal Kombat II',
 'Desire',
 'Zootopia 2',
 'The Super Mario

In [75]:
search_terms = ["spider", "interstellar", "inception", "martian"]

for term in search_terms:
    print(f"\n--- {term.upper()} ---")
    
    matches = enriched_df[
        enriched_df["title"].str.contains(
            term,
            case=False,
            na=False
        )
    ]
    
    print(matches["title"].tolist())


--- SPIDER ---
['Spider-Man: Brand New Day', 'Spider-Man: No Way Home', 'Spider-Man: Homecoming', 'Spider-Man', 'Spider-Man: Far From Home', 'The Amazing Spider-Man', 'The Amazing Spider-Man 2', 'Spider-Man 3']

--- INTERSTELLAR ---
[]

--- INCEPTION ---
[]

--- MARTIAN ---
[]


In [77]:
def recommend_for_user(liked_movies, top_n=5):
    
    movie_indices = []
    
    # Find indexes of liked movies
    for movie in liked_movies:
        
        matches = enriched_df[
            enriched_df["title"].str.strip().str.lower() == movie.strip().lower()
        ].index
        
        if len(matches) > 0:
            movie_indices.append(matches[0])
        else:
            print(f"Movie not found: {movie}")
    
    
    if len(movie_indices) == 0:
        print("No valid movies found.")
        return
    
    
    # Get TF-IDF vectors of liked movies
    liked_vectors = tfidf_matrix[movie_indices]
    
    
    # Create user taste vector
    user_vector = np.asarray(
        liked_vectors.mean(axis=0)
    )
    
    
    # Compare user taste with every movie
    user_similarity = cosine_similarity(
        user_vector,
        tfidf_matrix
    ).flatten()
    
    
    # Rank movies
    ranked_indices = np.argsort(user_similarity)[::-1]
    
    
    recommendations = []
    
    for index in ranked_indices:
        
        # Don't recommend movies already liked
        if index in movie_indices:
            continue
        
        recommendations.append({
            "title": enriched_df.iloc[index]["title"],
            "similarity_score": round(
                user_similarity[index] * 100,
                2
            )
        })
        
        if len(recommendations) == top_n:
            break
    
    
    return pd.DataFrame(recommendations)

In [78]:
recommend_for_user(
    [
        "Spider-Man: Brand New Day",
        "Spider-Man: No Way Home",
        "Spider-Man: Homecoming"
    ],
    top_n=5
)

,title,similarity_score
0,Spider-Man: Far From Home,40.69
1,The Amazing Spider-Man 2,31.53
2,The Amazing Spider-Man,29.87
3,Spider-Man,28.80
4,The Avengers,21.83
